# Módulo 02: Calidad de datos y valores faltantes

[Abrir en Colab](https://colab.research.google.com/github/sgevatschnaider/data-science-business-decisions/blob/main/notebooks/02-calidad-datos.ipynb)

**Pregunta de decisión:** ¿La evidencia disponible representa el proceso real o sus defectos pueden cambiar la conclusión y perjudicar la decisión?

**Autor:** Sergio Gevatschnaider


## Objetivos

- Definir pruebas de calidad alineadas con reglas de negocio.
- Distinguir ausencia MCAR, MAR y MNAR como hipótesis de trabajo.
- Comparar eliminación, imputación simple y estrategias multivariadas.
- Documentar impacto, trazabilidad e indicadores de ausencia.

**Criterio de éxito:** el resultado debe cambiar o sostener una acción concreta, superar una referencia y declarar límites.


## 1. Entorno reproducible

Registramos versiones y semilla antes de producir evidencia. Ejecutá siempre **Runtime → Run all** en Colab.


In [ ]:
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

SEED = 42
np.random.seed(SEED)
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})

## 2. Experimento base

El bloque siguiente construye una referencia mínima y verificable. No representa todavía la recomendación final.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

rng = np.random.default_rng(42)
datos = pd.DataFrame({
    "ocupacion": rng.choice(["Dependiente", "Independiente"], 200),
    "ingreso": rng.lognormal(11, 0.4, 200),
})
mask = (datos["ocupacion"].eq("Independiente") & (rng.random(200) < 0.35)) | (rng.random(200) < 0.05)
datos.loc[mask, "ingreso"] = np.nan
datos["ingreso_faltante"] = datos["ingreso"].isna().astype(int)
datos.groupby("ocupacion")["ingreso_faltante"].mean()

## 3. Evidencia visual

Una visualización útil permite comparar, muestra unidades y deja visible la incertidumbre o variación relevante.


In [ ]:
import matplotlib.pyplot as plt

faltantes = datos.groupby("ocupacion")["ingreso_faltante"].mean().sort_values()
ax = faltantes.mul(100).plot.barh(figsize=(7, 3), color="#b45309")
ax.set(xlabel="Ingreso faltante (%)", ylabel="Ocupación", title="Ausencia condicionada por ocupación")
plt.tight_layout()

## 4. Comparación para decidir

En una solicitud de crédito faltan ingresos con mayor frecuencia en trabajadores independientes. Imputar sin segmentar puede ocultar una diferencia estructural.

La tabla fuerza una comparación entre alternativas, costos o criterios. Adaptala a las unidades del caso.


In [ ]:
pd.DataFrame({'estrategia': ['eliminar', 'mediana global', 'mediana por ocupación'], 'riesgo': ['sesgo por selección', 'oculta heterogeneidad', 'requiere validar estabilidad'], 'uso': ['solo baja ausencia', 'baseline', 'comparación recomendada']})

## 5. Desafío de transferencia

**Definí qué defectos bloquean una decisión, cuáles admiten corrección y cuáles exigen recolectar datos nuevamente.**

1. Definir clave, rangos y reglas de consistencia.
2. Medir faltantes por variable y por segmento.
3. Crear indicadores de ausencia antes de imputar.
4. Comparar distribución y métrica posterior a cada tratamiento.

Antes de continuar, escribí una hipótesis, una condición que la refutaría y el costo de una decisión equivocada.

### Registro de decisión

Completá la celda siguiente como evidencia de cierre del laboratorio.


In [ ]:
decision_record = {
    "pregunta": '¿La evidencia disponible representa el proceso real o sus defectos pueden cambiar la conclusión y perjudicar la decisión?',
    "hipotesis": "Completar antes del análisis",
    "evidencia": "Registrar la tabla o visualización que cambia la decisión",
    "recomendacion": "Expresar acción, población y horizonte",
    "limitacion": "Indicar qué podría invalidar la conclusión",
    "responsable": "Asignar dueño y fecha de revisión",
}
pd.Series(decision_record, name="registro_de_decision")

## 6. Cierre verificable

**Entregable:** Reporte de calidad con reglas automatizadas, mapa de faltantes, hipótesis del mecanismo y comparación de dos estrategias.

- Hallazgo principal:
- Evidencia que lo respalda:
- Comparación contra baseline o escenario alternativo:
- Limitación:
- Acción, responsable y fecha de revisión:

Material elaborado por el profesor Sergio Gevatschnaider.
